# 🚁 Dunakeszi Comprehensive Test Notebook

**Purpose:** End-to-end evaluation of the drone detection & localisation pipeline across all test data sources.

| Section | What is tested |
|---------|----------------|
| 0 | Setup, Drive mount, package import |
| 1 | Config & model loading sanity checks |
| 2 | Ground-truth integrity (GT tables, timing, MEMS flags) |
| 3 | Extractor unit tests (trajectory, bearing helpers) |
| 4 | Pipeline adapter (azimuth convention, label conversion) |
| 5 | Dunakeszi **Professional mic (BK-6-E)** - zip OR folder input |
| 6 | Dunakeszi **MEMS** - detection-only (shows 5, 7, partial 8) |
| 7 | **External audio** (YouTube / any URL or uploaded file) |
| 8 | **Multi-drone** mode on formation segments |
| 9 | Regression dashboard - aggregate metrics across all sources |

> **How to use:**  
> Run 0 and 1 first (always). Then run any section independently.  
> Each section prints its own summary and saves CSVs to `cfg.DRIVE_LOGS`.

## 0 - Setup

In [ ]:
# - 0-A: Install dependencies ------------------------
# Run once per Colab session.
!pip install -q librosa soundfile pydub torch torchvision scipy yt-dlp
print('✅ Dependencies installed')

In [ ]:
# - 0-B: Mount Google Drive -------------------------
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# - 0-C: Import pipeline package ----------------------
import sys, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = '/content/drive/MyDrive/drone_detection'
sys.path.insert(0, PROJECT_ROOT)

# Core pipeline - reuse existing class definitions (no re-implementation)
from drone_detection import (
    config, Config,
    DetectionCNN, LocalizationCNN, make_localization_model,
    load_detection_model, load_localization_model,
    detect, localize, load_3ch, run_pipeline,
    analyse_audio_file, analyse_external_audio_robust,
    heuristic_detect,
    KalmanTrack, KalmanTracker,
    localize_multi_drone,
    AudioProcessor, synthesise_drone,
    compute_ipd_features, angular_error_deg,
    plot_polar_azimuth, plot_multi_drone_positions,
    plot_track_trajectory,
)

from drone_detection.mems_inference import (
    analyse_mems_file, batch_analyse_mems,
    analyse_mems_with_meta, build_mems_report,
)

from drone_detection.inference_test_loader import (
    load_test_dataset_zip, run_test_dataset_evaluation, TestDataset, TestSession,
)

# Dunakeszi-specific scripts (assumed to live alongside this notebook or in PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT + '/scripts')  # adjust if needed
from prepare_dunakeszi_for_pipeline import (
    bearing_to_pipeline_az, angular_diff, convert_label, prepare_dunakeszi, verify_output,
)
from dunakeszi_ground_truth_fixed import (
    load_ground_truth, segments_for_file, timing_for_segment,
    polywav_seek_sample, mems_seek_sample,
    gps_to_xy, build_mic_geometry,
)
from dunakeszi_segment_extractor_fixed import (
    interpolate_position, compute_clip_trajectory, _pos_to_bearing,
    ARRAY_CHANNELS, NATIVE_SR, TARGET_SR,
)
from dunakeszi_segment_analysis_inference import run_segment_analysis

import numpy as np
import pandas as pd
import math, json, zipfile, shutil, tempfile
from pathlib import Path
from IPython.display import display

print('✅ All imports OK')
print(f'   Config: SR={config.SR} Hz, N_MELS={config.N_MELS}, device={config.DEVICE}')

In [ ]:
# - 0-D: Path configuration -------------------------
# Edit these to match your Drive layout.
DRIVE_BASE   = Path('/content/drive/MyDrive/dunakeszi_2025')

# Ground-truth tables (output of dunakeszi_ground_truth_fixed.py)
GT_DIR       = DRIVE_BASE / 'ground_truth'

# Pipeline-ready extracted segments (output of prepare_dunakeszi_for_pipeline.py)
# Either a directory OR a .zip - both are handled by 5
PROF_MIC_DIR  = DRIVE_BASE / 'dunakeszi_pipeline_ready'        # BK-6-E segments
PROF_MIC_ZIP  = DRIVE_BASE / 'dunakeszi_pipeline_ready.zip'    # same, zipped

# MEMS segments (single-channel, extracted separately)
MEMS_DIR     = DRIVE_BASE / 'dunakeszi_mems_clips'

# External audio
EXTERNAL_DIR = DRIVE_BASE / 'external_audio'
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

# All results go here
RESULTS_DIR  = config.DRIVE_LOGS / 'dunakeszi_tests'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Results → {RESULTS_DIR}')

## 1 - Config & Model Loading

In [ ]:
# - 1-A: Apply Dunakeszi config patch -------------------─
# Required before running any Dunakeszi inference.
config.set_array_geometry('gp2')          # BK-6-E: 2500 mm equilateral triangle
config.MAX_LOCALIZATION_DIST = 100.0      # covers all orbit radii up to 60 m

print('Array geometry:', config.ARRAY_GEOMETRY)
print('MIC_POSITIONS (metres):')
print(config.MIC_POSITIONS)
print(f'MAX_LOCALIZATION_DIST = {config.MAX_LOCALIZATION_DIST} m')
print(f'BPF_ENERGY_RATIO_AS_FEATURE = {config.BPF_ENERGY_RATIO_AS_FEATURE}')

In [ ]:
# - 1-B: Load models ----------------------------─
det_model = load_detection_model(config)
try:
    loc_model = load_localization_model(config)
    CAN_LOCALIZE = True
except FileNotFoundError as e:
    print(f'⚠️  No localization model: {e}')
    print('   Detection-only mode will be used for localization sections.')
    CAN_LOCALIZE = False

# Verify model architecture
import torch
total_det = sum(p.numel() for p in det_model.parameters())
print(f'\nDetectionCNN  parameters : {total_det:,}')
if CAN_LOCALIZE:
    total_loc = sum(p.numel() for p in loc_model.parameters())
    print(f'LocalizationCNN parameters: {total_loc:,}')

# Quick smoke test with a synthetic signal
with torch.no_grad():
    dummy_mel = torch.zeros(1, 3, config.N_MELS, 26)
    dummy_ipd = torch.zeros(1, 4 if config.BPF_ENERGY_RATIO_AS_FEATURE else 3)
    logits = det_model(dummy_mel)
    assert logits.shape == (1, 2), f'Unexpected shape: {logits.shape}'
    if CAN_LOCALIZE:
        out = loc_model(dummy_mel, dummy_ipd)
        assert out.shape == (1, 4), f'Unexpected shape: {out.shape}'
print('\n✅ Model smoke test passed')

## 2 - Ground-Truth Integrity

In [ ]:
# - 2-A: Load ground-truth tables ---------------------─
if not GT_DIR.exists():
    print(f'⚠️  GT_DIR not found: {GT_DIR}')
    print('   Run dunakeszi_ground_truth_fixed.py --out_dir ground_truth/ first.')
    GT = None
else:
    GT = load_ground_truth(str(GT_DIR))
    segs = GT['segments']
    print(f'✅ Ground truth loaded')
    print(f'   Sessions : {len(GT["sessions"])}')
    print(f'   Segments : {len(segs)}')

    total_s = sum(s['duration_s'] for s in segs)
    print(f'   Total audio: {total_s:.0f}s ({total_s/60:.1f} min)')

    for sp in ('train', 'val', 'test'):
        sub = [s for s in segs if s['split'] == sp]
        dur = sum(s['duration_s'] for s in sub)
        print(f'     {sp:5s}: {len(sub):2d} segments, {dur:.0f}s ({dur/60:.1f} min)')

    mems_segs = [s for s in segs if s.get('mems_available')]
    print(f'   MEMS available: {len(mems_segs)} segments')

In [ ]:
# - 2-B: Segment integrity checks ---------------------─
if GT is not None:
    issues = []
    for seg in GT['segments']:
        sid = seg['id']
        # Onset must be non-negative
        if seg['onset_from_rec_s'] < 0:
            issues.append(f'  seg {sid}: negative onset {seg["onset_from_rec_s"]}')
        # Duration must be positive
        if seg['duration_s'] <= 0:
            issues.append(f'  seg {sid}: non-positive duration {seg["duration_s"]}')
        # Split must be valid
        if seg.get('split') not in ('train', 'val', 'test', None):
            issues.append(f'  seg {sid}: unknown split "{seg.get("split")}"')

    if issues:
        print(f'⚠️  {len(issues)} integrity issues:')
        for i in issues[:20]:
            print(i)
    else:
        print('✅ All segment records pass integrity checks')

    # MEMS coverage - expected only shows 5, 7, and partial 8
    mems_sessions = set(s.get('session') for s in mems_segs)
    print(f'\n   MEMS sessions: {sorted(mems_sessions)}')
    print('   Expected: show_5, show_7, (partial show_8)')

In [ ]:
# - 2-C: Mic array geometry verification ------------------
mics = build_mic_geometry()
print('Mic array geometry (XY metres from origin):')
for name, info in mics['arrays'].items():
    xy = info.get('xy_m', '-')
    gps = info.get('gps', {})
    print(f'  {name:8s}: xy={xy}  gps=({gps.get("lat","?"):.7f}, {gps.get("lon","?"):.7f})')

# Cross-check BK-6-E baseline against config
bke = mics['arrays'].get('BK-6-E', {})
bkw = mics['arrays'].get('BK-6-W', {})
if 'xy_m' in bke and 'xy_m' in bkw:
    baseline = np.linalg.norm(np.array(bke['xy_m']) - np.array(bkw['xy_m']))
    print(f'\nBK-E to BK-W baseline: {baseline:.2f} m  (expected ≈ 2.0 m east-west)')

## 3 - Extractor Unit Tests

In [ ]:
# - 3-A: interpolate_position() for all maneuver types ----------─
print('Testing interpolate_position() for all maneuver types')
print('=' * 60)

maneuver_cases = [
    # (maneuver_type, seg_dict_extra, t_within, expected_description)
    ('hover',    {'start_coord': [30.0, 0.0, 40.0]},                        1.5,  'hover: position unchanged'),
    ('transit',  {'start_coord': [0.0, 0.0, 40.0], 'end_coord': [60.0, 0.0, 40.0], 'duration_s': 10.0}, 5.0, 'transit: midpoint ≈ 30 m East'),
    ('circle',   {'start_coord': [30.0, 0.0, 40.0], 'radius_m': 30.0, 'speed_mps': 4.0, 'duration_s': 47.1}, 0.0, 'circle: starts at start_coord'),
    ('figure8',  {'start_coord': [0.0, 30.0, 40.0], 'radius_m': 30.0, 'speed_mps': 4.0, 'duration_s': 94.2}, 0.0, 'figure8: starts near start_coord'),
    ('diagonal', {'start_coord': [-30.0, -30.0, 40.0], 'end_coord': [30.0, 30.0, 40.0], 'duration_s': 10.0}, 5.0, 'diagonal: midpoint ≈ origin'),
    ('survey',   {'start_coord': [20.0, 10.0, 50.0]},                       99.0, 'survey: position frozen'),
]

all_ok = True
for mtype, extra, t, desc in maneuver_cases:
    seg = {'maneuver_type': mtype, 'duration_s': extra.get('duration_s', 10.0), **extra}
    x, y, z = interpolate_position(seg, t)
    bearing  = _pos_to_bearing(x, y, z)
    ok = all(math.isfinite(v) for v in [x, y, z])
    status = '✅' if ok else '❌'
    if not ok: all_ok = False
    print(f'  {status} {mtype:10s}  t={t:5.1f}s → x={x:7.2f} y={y:7.2f} z={z:5.1f}  '
          f'az={bearing["azimuth_deg"]:+7.1f}°  d={bearing["distance_xy_m"]:5.1f}m  # {desc}')

print()
print('✅ All trajectory types OK' if all_ok else '❌ Some trajectory types failed')

In [ ]:
# - 3-B: compute_clip_trajectory() - 3-second clip in a circle ------─
print('Testing compute_clip_trajectory() on a 30m-radius circle segment')
circle_seg = {
    'maneuver_type': 'circle',
    'start_coord':   [30.0, 0.0, 40.0],
    'radius_m':      30.0,
    'speed_mps':     4.0,
    'duration_s':    47.1,
    'altitude_m':    40.0,
}
traj = compute_clip_trajectory(circle_seg, clip_start_s=0.0, clip_dur_s=3.0)

assert 'samples' in traj,       'Missing samples key'
assert len(traj['samples']) == 30, f'Expected 30 samples at 10 Hz, got {len(traj["samples"])}'
assert 'azimuth_swept_deg' in traj, 'Missing azimuth_swept_deg'

print(f'  Samples          : {len(traj["samples"])} (@ 10 Hz, 3 s)')
print(f'  Azimuth swept    : {traj["azimuth_swept_deg"]:+.2f}°')
print(f'  Start az/dist    : {traj["azimuth_start_deg"]:+.1f}°  {traj["distance_start_m"]:.1f}m')
print(f'  Mid   az/dist    : {traj["azimuth_mid_deg"]:+.1f}°  {traj["distance_mid_m"]:.1f}m')
print(f'  End   az/dist    : {traj["azimuth_end_deg"]:+.1f}°  {traj["distance_end_m"]:.1f}m')

# Full-segment mode (clip_dur_s=0)
traj_full = compute_clip_trajectory(circle_seg, clip_start_s=0.0, clip_dur_s=0)
print(f'  Full-seg samples : {len(traj_full["samples"])} (dur={circle_seg["duration_s"]}s)')
print('✅ compute_clip_trajectory OK')

## 4 - Pipeline Adapter (Azimuth Convention)

In [ ]:
# - 4-A: bearing_to_pipeline_az() - all 8 compass directions --------
# Convention: bearing_from_north  → pipeline math angle (atan2 y,x, from East)
# Conversion: pipeline_az = 90 - bearing
print('Azimuth conversion: bearing-from-North → pipeline math angle')
print('=' * 65)

compass_cases = [
    # (x_m, y_m, direction, expected_bearing, expected_pipeline_az)
    ( 0.0,  60.0, 'North',   0.0,    90.0),   # pipeline East=+90 equiv - wait, N→pipeline=90
    ( 60.0,  0.0, 'East',   90.0,     0.0),   # East → pipeline 0°
    ( 0.0, -60.0, 'South',  180.0,  -90.0),   # or +270, wrapped → -90
    (-60.0,  0.0, 'West',   -90.0,  180.0),   # or -180
    ( 60.0,  60.0, 'NE',    45.0,    45.0),   # diagonal unchanged
    (-60.0,  60.0, 'NW',   -45.0,  135.0),
    ( 60.0, -60.0, 'SE',   135.0,   -45.0),
    (-60.0, -60.0, 'SW',  -135.0, -135.0),   # diagonal unchanged
]

all_ok = True
print(f'  {"Direction":6s}  {"Bearing":>8s}  {"Pipeline":>8s}  {"Expected":>8s}  {"AngDiff":>7s}  Status')
print('  ' + '-'*60)
for x, y, direction, bearing_gt, expected_pipeline in compass_cases:
    # Verify bearing from coords via _pos_to_bearing
    bearing_from_coords = math.degrees(math.atan2(x, y))   # extractor convention
    pipeline_az = bearing_to_pipeline_az(bearing_from_coords)
    diff = angular_diff(pipeline_az, expected_pipeline)
    ok = diff < 0.5   # within 0.5° tolerance
    if not ok: all_ok = False
    status = '✅' if ok else f'❌ (diff={diff:.1f}°)'
    print(f'  {direction:6s}  {bearing_from_coords:+8.1f}°  {pipeline_az:+8.1f}°  '
          f'{expected_pipeline:+8.1f}°  {diff:7.2f}°  {status}')

print()
print('✅ All compass directions convert correctly' if all_ok else '❌ Some conversions failed')

In [ ]:
# - 4-B: convert_label() - full round-trip test --------------
print('convert_label() round-trip tests')
print('=' * 60)

MAX_DIST = 100.0

test_raw_labels = [
    # (raw_dict, description)
    ({'drone': {'azimuth': 45.0, 'distance': 30.0, 'height': 40.0},
      'segment_id': 'test_NE', 'session': 'show_5', 'maneuver_type': 'circle',
      'n_drones': 1, 'split': 'test', 'array': 'BK-6-E'},
     'NE orbit, 30m radius'),

    ({'drone': {'azimuth': -90.0, 'distance': 60.0, 'height': 50.0},
      'segment_id': 'test_W', 'session': 'show_7', 'maneuver_type': 'circle',
      'n_drones': 1, 'split': 'test', 'array': 'BK-6-E'},
     'West orbit, 60m radius'),

    ({'drone': {'azimuth': 0.0, 'distance': 0.0, 'height': 0.0},
      'segment_id': 'test_fig8', 'session': 'show_8', 'maneuver_type': 'figure8',
      'n_drones': 1, 'split': 'test', 'array': 'BK-6-E'},
     'figure-8 (no position)'),
]

for raw, desc in test_raw_labels:
    converted = convert_label(raw, MAX_DIST)
    has_pos   = converted['has_position']
    az        = converted['azimuth_deg']
    dist      = converted['distance_m']
    ht        = converted['height_m']

    # Key presence check
    for k in ('azimuth_deg', 'distance_m', 'height_m', 'source', 'has_position'):
        assert k in converted, f'Missing key: {k}'

    if has_pos:
        assert az is not None and -180 <= az <= 180, f'az={az} out of range'
        assert dist is not None and dist >= 0, f'dist={dist} invalid'
    
    print(f'  [{desc}]')
    print(f'    has_position={has_pos}  az={az}  dist={dist}  ht={ht}')
    print(f'    note: {converted["note"][:80]}')

print('\n✅ convert_label() round-trip OK')

In [ ]:
# - 4-C: verify_output() on existing pipeline-ready directory -------─
if PROF_MIC_DIR.exists():
    verify_output(PROF_MIC_DIR, max_dist=100.0)
elif PROF_MIC_ZIP.exists():
    # Extract zip to temp and verify
    import tempfile, zipfile, shutil
    tmp = Path(tempfile.mkdtemp(prefix='dkz_verify_'))
    with zipfile.ZipFile(PROF_MIC_ZIP, 'r') as zf:
        zf.extractall(tmp)
    # Find the extracted folder
    subdirs = [p for p in tmp.iterdir() if p.is_dir()]
    verify_root = subdirs[0] if subdirs else tmp
    verify_output(verify_root, max_dist=100.0)
    shutil.rmtree(tmp)
else:
    print(f'⚠️  Neither {PROF_MIC_DIR} nor {PROF_MIC_ZIP} found - skipping verify_output()')

## 5 - Dunakeszi Professional Mic (BK-6-E) - Detection & Localisation

In [ ]:
# - 5-A: Input source selection ----------------------─
# Choose ONE of the three options below by setting PROF_MIC_SOURCE.
#
#   'dir'    - use PROF_MIC_DIR (already extracted folder)
#   'zip'    - use PROF_MIC_ZIP (will be extracted automatically)
#   'upload' - upload a zip or individual files via Colab file picker

PROF_MIC_SOURCE = 'zip'   # ← change as needed

_prof_zip_path  = None
_prof_dir_path  = None

if PROF_MIC_SOURCE == 'dir':
    if not PROF_MIC_DIR.exists():
        raise FileNotFoundError(f'PROF_MIC_DIR not found: {PROF_MIC_DIR}')
    _prof_dir_path = PROF_MIC_DIR
    print(f'Using directory: {_prof_dir_path}')

elif PROF_MIC_SOURCE == 'zip':
    if not PROF_MIC_ZIP.exists():
        raise FileNotFoundError(f'PROF_MIC_ZIP not found: {PROF_MIC_ZIP}')
    _prof_zip_path = str(PROF_MIC_ZIP)
    print(f'Using zip: {_prof_zip_path}')

elif PROF_MIC_SOURCE == 'upload':
    from google.colab import files as _colab_files
    print('📁 Upload your dunakeszi_pipeline_ready ZIP or individual WAV/JSON files...')
    _uploaded = _colab_files.upload()
    _zips = [k for k in _uploaded if k.lower().endswith('.zip')]
    _wavs = [k for k in _uploaded if k.lower().endswith('.wav')]

    if _zips:
        _prof_zip_path = _zips[0]
        print(f'ZIP detected: {_prof_zip_path}')
    elif _wavs:
        # Individual files - move them to a temp dir
        _prof_dir_path = Path(tempfile.mkdtemp(prefix='dkz_upload_'))
        for fname in _uploaded:
            (_prof_dir_path / fname).write_bytes(_uploaded[fname])
        print(f'Individual files staged in: {_prof_dir_path}  ({len(_uploaded)} files)')
    else:
        raise ValueError('No ZIP or WAV files found in upload.')
else:
    raise ValueError(f'Unknown PROF_MIC_SOURCE={PROF_MIC_SOURCE!r}')

In [ ]:
# - 5-B: Full evaluation via load_test_dataset_zip ------------─
# This reuses the existing load_test_dataset_zip / run_test_dataset_evaluation
# API - no duplication of logic.

PROF_MIC_RESULTS = None

if _prof_zip_path:
    print('🔄 Loading Dunakeszi Professional Mic dataset from ZIP...')
    prof_test_ds = load_test_dataset_zip(
        zip_path       = _prof_zip_path,
        cfg            = config,
        dataset_format = 'generic_triplet',   # *_ch0/1/2.wav + *_label.json
    )

elif _prof_dir_path:
    # Build a TestDataset manually from the directory
    from drone_detection.inference_test_loader import TestSession, TestDataset
    import json as _json

    _label_files = sorted(Path(_prof_dir_path).glob('*_label.json'))
    _sessions = []
    for lf in _label_files:
        stem = lf.stem.replace('_label', '')
        wavs = [str(_prof_dir_path / f'{stem}_ch{i}.wav') for i in range(3)]
        if not all(Path(w).exists() for w in wavs):
            continue
        label = _json.loads(lf.read_text())
        _sessions.append(TestSession(
            session_id  = stem,
            wav_paths   = wavs,
            azimuth_deg = label.get('azimuth_deg'),
            distance_m  = label.get('distance_m'),
            height_m    = label.get('height_m'),
            metadata    = label,
        ))
    n_lab   = sum(1 for s in _sessions if s.has_label)
    prof_test_ds = TestDataset(
        sessions       = _sessions,
        source_zip     = str(_prof_dir_path),
        dataset_format = 'generic_triplet',
        extract_dir    = str(_prof_dir_path),
        n_labelled     = n_lab,
        n_unlabelled   = len(_sessions) - n_lab,
    )
    print(f'✅ Indexed {len(prof_test_ds)} sessions from directory ({n_lab} labelled)')

if prof_test_ds:
    PROF_MIC_RESULTS = run_test_dataset_evaluation(
        test_ds    = prof_test_ds,
        cfg        = config,
        show_plots = True,
        save_csv   = str(RESULTS_DIR / 'prof_mic_results.csv'),
    )

In [ ]:
# - 5-C: Per-maneuver-type breakdown --------------------
if PROF_MIC_RESULTS and PROF_MIC_RESULTS.get('sessions'):
    df_prof = pd.DataFrame(PROF_MIC_RESULTS['sessions'])

    # Merge maneuver_type from label metadata via session_id
    id_to_meta = {s.session_id: s.metadata for s in prof_test_ds.sessions}
    df_prof['maneuver_type'] = df_prof['session_id'].map(
        lambda sid: id_to_meta.get(sid, {}).get('maneuver_type', 'unknown')
    )

    print('Per-maneuver detection and localisation accuracy:')
    print('=' * 70)
    grp = df_prof.groupby('maneuver_type').agg(
        n          = ('session_id', 'count'),
        det_rate   = ('detected', 'mean'),
        mae_az     = ('az_err_deg', 'mean'),
        mae_dist   = ('dist_err_m', 'mean'),
        mae_ht     = ('ht_err_m', 'mean'),
    ).round(2)
    display(grp)

    # Split-level breakdown
    df_prof['split'] = df_prof['session_id'].map(
        lambda sid: id_to_meta.get(sid, {}).get('split', 'unknown')
    )
    print('\nPer-split detection rate:')
    display(df_prof.groupby('split')[['detected', 'az_err_deg', 'dist_err_m']].mean().round(3))
else:
    print('⚠️  No results available - run 5-B first.')

In [ ]:
# - 5-D: run_segment_analysis() - full dashboard with Kalman tracking ----
# This uses the dunakeszi_segment_analysis_inference.py API directly.

if _prof_dir_path and _prof_dir_path.exists():
    print('🔄 Running segment analysis with Kalman tracking...')
    seg_results = run_segment_analysis(
        segment_dir  = str(_prof_dir_path),
        cfg          = config,
        file_ext     = ['.wav'],
        sort_by_name = True,
        n_mics       = 3,
        show_plots   = True,
        save_plots   = True,
        save_csv     = True,
    )
    print(f'\n📊 Detections: {seg_results["n_detected"]}/{seg_results["n_segments"]}')
    print(f'   Confirmed tracks: {len(seg_results["confirmed_tracks"])}')
else:
    print('⚠️  Directory mode required for run_segment_analysis - set PROF_MIC_SOURCE="dir" or "upload".')

## 6 - Dunakeszi MEMS Recording (Detection Only)

In [ ]:
# - 6-A: MEMS availability flag ----------------------─
# MEMS only covers show_5 (14:08), show_7 (14:24), and partial show_8 (14:41).
# Set SKIP_MEMS = True if you don't have MEMS clips available.
SKIP_MEMS = not MEMS_DIR.exists()

if SKIP_MEMS:
    print(f'⚠️  MEMS_DIR not found: {MEMS_DIR}')
    print('   Set MEMS_DIR to your MEMS clip folder to enable 6.')
    print('   Skipping all MEMS tests.')
else:
    mems_files = sorted(MEMS_DIR.glob('*.wav'))
    print(f'✅ MEMS directory found: {len(mems_files)} WAV clips')
    print('   Note: azimuth/distance localisation is NOT available for MEMS (single-channel).')

In [ ]:
# - 6-B: MEMS input - upload or directory -----------------─
# You can also upload MEMS clips individually from Colab if MEMS_DIR doesn't exist.

if SKIP_MEMS:
    print('Upload MEMS WAV clips to enable MEMS testing:')
    print('  from google.colab import files')
    print('  uploaded = files.upload()   # select your _mems_*.wav files')
    print('  Then set SKIP_MEMS = False and MEMS_DIR to the upload location.')
else:
    print(f'Using MEMS clips from: {MEMS_DIR}')
    for f in mems_files[:5]:
        print(f'  {f.name}')
    if len(mems_files) > 5:
        print(f'  ... and {len(mems_files) - 5} more')

In [ ]:
# - 6-C: Batch MEMS detection -----------------------─
MEMS_RESULTS = None

if not SKIP_MEMS:
    print('🔄 Running batch MEMS detection (detection-only mode)...')
    MEMS_RESULTS = batch_analyse_mems(
        folder     = str(MEMS_DIR),
        cfg        = config,
        show_plots = False,    # set True for per-file dashboards
    )
    mems_report = build_mems_report(MEMS_RESULTS)
    print(mems_report)

In [ ]:
# - 6-D: MEMS with ground-truth meta (if available) ------------─
# Segments that have a companion _meta.json can be cross-checked against GT.
if not SKIP_MEMS:
    meta_files = sorted(MEMS_DIR.glob('*_meta.json'))
    if not meta_files:
        print('ℹ️  No *_meta.json found - skipping GT crosscheck.')
        print('   Add a _meta.json alongside each MEMS clip to enable GT comparison.')
    else:
        print(f'Found {len(meta_files)} meta files - running with GT annotations...')
        meta_results = []
        for wav_f in mems_files:
            meta_f = wav_f.with_name(wav_f.stem + '_meta.json')
            if meta_f.exists():
                r = analyse_mems_with_meta(
                    wav_path   = str(wav_f),
                    meta_path  = str(meta_f),
                    cfg        = config,
                    show_plot  = False,
                )
                meta_results.append(r)
        if meta_results:
            df_mems = pd.DataFrame(meta_results)
            print('MEMS detection with GT crosscheck:')
            display(df_mems[['filename', 'detected', 'probability', 'gt_label', 'correct']].head(30))
            acc = df_mems['correct'].mean()
            print(f'\nMEMS detection accuracy: {acc:.1%}  ({df_mems["correct"].sum()}/{len(df_mems)})')

In [ ]:
# - 6-E: MEMS heuristic-only analysis (no model needed) ----------─
if not SKIP_MEMS:
    import soundfile as sf
    ap = AudioProcessor(config)
    print('Heuristic-only analysis on MEMS clips:')
    print(f'{"File":45s}  {"Heur_P":7s}  {"Label":10s}')
    print('-' * 70)
    for wav_f in mems_files[:20]:   # cap at 20 for display
        y, sr = sf.read(str(wav_f), dtype='float32')
        if y.ndim > 1: y = y[:, 0]
        import librosa as _lib
        y_r = _lib.resample(y, orig_sr=sr, target_sr=config.SR)
        y_r = ap.pad_or_truncate(y_r)
        h = heuristic_detect(y_r, config)
        print(f'{wav_f.name:45s}  {h["probability"]:7.3f}  {h["label"]:10s}')

## 7 - External Audio (YouTube / URL / File Upload)

In [ ]:
# - 7-A: Configure external audio sources -----------------─
# Add YouTube URLs, direct audio URLs, or leave empty to upload files below.
# The pipeline downloads, converts to mono WAV, and runs sliding-window analysis.

EXTERNAL_SOURCES = [
    # Example drone flyby videos - replace or add your own:
    # {'label': 'DJI Mavic flyby',   'url': 'https://www.youtube.com/watch?v=XXXXXXXXXXX'},
    # {'label': 'DJI Phantom hover', 'url': 'https://www.youtube.com/watch?v=YYYYYYYYYYY'},
]

# Segment length and overlap for sliding-window analysis
EXTERNAL_SEGMENT_SEC = 3.0
EXTERNAL_OVERLAP     = 0.5    # 50% overlap
EXTERNAL_THRESHOLD   = config.DETECTION_THRESHOLD

print(f'External sources configured: {len(EXTERNAL_SOURCES)}')
print(f'Segment: {EXTERNAL_SEGMENT_SEC}s  Overlap: {EXTERNAL_OVERLAP:.0%}  Threshold: {EXTERNAL_THRESHOLD:.3f}')

In [ ]:
# - 7-B: Upload audio files (alternative to URLs) -------------─
# Upload WAV / MP3 / M4A / FLAC files directly from your machine.

USE_FILE_UPLOAD = False   # ← set True to activate Colab file picker

if USE_FILE_UPLOAD:
    from google.colab import files as _cf
    print('📁 Select audio files to analyse...')
    _uploaded_ext = _cf.upload()
    for fname, data in _uploaded_ext.items():
        dest = EXTERNAL_DIR / fname
        dest.write_bytes(data)
        EXTERNAL_SOURCES.append({'label': fname, 'local_path': str(dest)})
    print(f'✅ {len(_uploaded_ext)} file(s) ready for analysis')

In [ ]:
# - 7-C: Download from URLs using yt-dlp ------------------
import subprocess

def download_audio(url: str, out_dir: Path, label: str) -> str:
    """Download audio from a URL or YouTube link using yt-dlp."""
    slug = label.replace(' ', '_').replace('/', '_')[:40]
    out_tmpl = str(out_dir / f'{slug}.%(ext)s')
    cmd = [
        'yt-dlp', '--no-playlist',
        '-x', '--audio-format', 'wav',
        '--audio-quality', '0',
        '-o', out_tmpl, url
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'yt-dlp failed: {result.stderr[:300]}')
    # Find the downloaded file
    candidates = sorted(out_dir.glob(f'{slug}.*'))
    if not candidates:
        raise FileNotFoundError(f'Downloaded file not found in {out_dir}')
    return str(candidates[-1])

EXTERNAL_AUDIO_READY = []   # populated below

for src in EXTERNAL_SOURCES:
    if 'local_path' in src:
        EXTERNAL_AUDIO_READY.append(src)
        continue
    url   = src.get('url', '')
    label = src.get('label', url[:40])
    if not url:
        continue
    try:
        print(f'⬇️  Downloading: {label} ...')
        local_path = download_audio(url, EXTERNAL_DIR, label)
        EXTERNAL_AUDIO_READY.append({'label': label, 'url': url, 'local_path': local_path})
        print(f'   ✅ Saved: {local_path}')
    except Exception as e:
        print(f'   ❌ Failed: {e}')

print(f'\n{len(EXTERNAL_AUDIO_READY)} source(s) ready for analysis')

In [ ]:
# - 7-D: Sliding-window analysis on each external file -----------
EXTERNAL_RESULTS = []

if not EXTERNAL_AUDIO_READY:
    print('ℹ️  No external audio sources configured.')
    print('   Add YouTube URLs to EXTERNAL_SOURCES or set USE_FILE_UPLOAD=True in 7-B.')
else:
    for src in EXTERNAL_AUDIO_READY:
        lpath = src.get('local_path', '')
        label = src.get('label', Path(lpath).name if lpath else 'unknown')
        if not lpath or not Path(lpath).exists():
            print(f'⚠️  {label}: file not found - skipping')
            continue
        print(f'\n🔍 Analysing: {label}')
        try:
            result = analyse_external_audio_robust(
                audio_path        = lpath,
                cfg               = config,
                segment_sec       = EXTERNAL_SEGMENT_SEC,
                overlap           = EXTERNAL_OVERLAP,
                threshold         = EXTERNAL_THRESHOLD,
                show_plot         = True,
            )
            result['label'] = label
            result['path']  = lpath
            EXTERNAL_RESULTS.append(result)

            n_det  = sum(1 for s in result.get('segments', []) if s.get('detected'))
            n_tot  = len(result.get('segments', []))
            max_p  = max((s.get('prob', 0) for s in result.get('segments', [])), default=0)
            print(f'   Detections: {n_det}/{n_tot}  peak_prob={max_p:.3f}')
        except Exception as e:
            print(f'   ❌ Error: {e}')

# Save summary
if EXTERNAL_RESULTS:
    rows = []
    for r in EXTERNAL_RESULTS:
        for seg in r.get('segments', []):
            rows.append({'label': r['label'], **seg})
    if rows:
        pd.DataFrame(rows).to_csv(RESULTS_DIR / 'external_audio_results.csv', index=False)
        print(f'\n💾 Saved: {RESULTS_DIR / "external_audio_results.csv"}')

In [ ]:
# - 7-E: run_segment_analysis() in mono (1-mic) mode on external audio ---
# Segments are already split by analyse_external_audio_robust; this cell
# demonstrates the full segment-analysis pipeline on a folder of pre-cut clips.

EXTERNAL_SEGMENTS_DIR = EXTERNAL_DIR / 'segments'

if EXTERNAL_SEGMENTS_DIR.exists() and any(EXTERNAL_SEGMENTS_DIR.glob('*.wav')):
    print('🔄 Running segment analysis in mono mode on external audio...')
    ext_seg_results = run_segment_analysis(
        segment_dir  = str(EXTERNAL_SEGMENTS_DIR),
        cfg          = config,
        file_ext     = ['.wav', '.mp3', '.flac'],
        sort_by_name = True,
        n_mics       = 1,       # mono / single-channel
        show_plots   = True,
        save_plots   = True,
        save_csv     = True,
    )
    print(f'Detections: {ext_seg_results["n_detected"]}/{ext_seg_results["n_segments"]}')
else:
    print(f'ℹ️  No pre-cut clips in {EXTERNAL_SEGMENTS_DIR} - skipping mono segment analysis.')

## 8 - Multi-Drone Mode (Formation Segments)

In [ ]:
# - 8-A: Synthetic multi-drone test --------------------─
# Generate 2-drone synthetic recordings and verify localize_multi_drone().

print('🚁🚁 Synthetic multi-drone localisation test')
print('=' * 60)

DRONE_PAIRS = [
    # ((x1, y1), (x2, y2), description)
    (( 30.0,  0.0), (-30.0,  0.0), 'East vs West   - 60m separation'),
    (( 20.0, 20.0), (-20.0, -20.0),'NE vs SW       - diagonal'),
    ((  0.0, 40.0), ( 30.0, -20.0),'North vs SE    - asymmetric'),
]

multi_drone_results = []
for (x1, y1), (x2, y2), desc in DRONE_PAIRS:
    # Synthesise two drones
    ch_d1 = synthesise_drone(config.MIC_POSITIONS, [x1, y1], cfg=config, noise_level=0.02)
    ch_d2 = synthesise_drone(config.MIC_POSITIONS, [x2, y2], cfg=config, noise_level=0.02)
    # Mix channels
    channels = [ch_d1[i] + ch_d2[i] for i in range(3)]

    # Run multi-drone localisation
    drones = localize_multi_drone(channels, config, max_drones=2)

    gt_az1 = math.degrees(math.atan2(y1, x1))
    gt_az2 = math.degrees(math.atan2(y2, x2))
    gt_d1  = math.hypot(x1, y1)
    gt_d2  = math.hypot(x2, y2)

    print(f'\n  [{desc}]')
    print(f'    GT: drone1 az={gt_az1:+.1f}° dist={gt_d1:.1f}m  |  '
          f'drone2 az={gt_az2:+.1f}° dist={gt_d2:.1f}m')
    print(f'    Found {len(drones)} drone(s):')
    for i, d in enumerate(drones):
        print(f'      [{i+1}] az={d["azimuth_deg"]:+6.1f}°  dist={d["distance_m"]:5.1f}m  '
              f'residual={d["tdoa_residual"]:.2e}  conf_r={d["confidence_radius"]:.2f}m')

    multi_drone_results.append({
        'description': desc,
        'n_found': len(drones),
        'drones': drones,
        'gt': [{'az': gt_az1, 'dist': gt_d1}, {'az': gt_az2, 'dist': gt_d2}],
    })

print('\n✅ Multi-drone synthetic test complete')

In [ ]:
# - 8-B: Multi-drone on real Dunakeszi formation segments ---------─
# Looks for segments with n_drones > 1 in the pipeline-ready directory.

formation_sessions = []
if _prof_dir_path and _prof_dir_path.exists():
    import json as _json
    for lf in sorted(_prof_dir_path.glob('*_label.json')):
        label = _json.loads(lf.read_text())
        if label.get('n_drones', 1) > 1:
            stem = lf.stem.replace('_label', '')
            wavs = [str(_prof_dir_path / f'{stem}_ch{i}.wav') for i in range(3)]
            if all(Path(w).exists() for w in wavs):
                formation_sessions.append({'stem': stem, 'label': label, 'wavs': wavs})

if not formation_sessions:
    print('ℹ️  No multi-drone formation segments found in the pipeline-ready directory.')
    print('   If your dataset has n_drones > 1 segments, ensure labels are correctly set.')
else:
    print(f'Found {len(formation_sessions)} formation segment(s):')
    for sess in formation_sessions:
        wavs = sess['wavs']
        label = sess['label']
        channels = load_3ch(wavs, config)

        det = detect(channels, config)
        drones = localize_multi_drone(channels, config, max_drones=label.get('n_drones', 2))

        print(f'\n  {sess["stem"]}')
        print(f'    GT n_drones={label.get("n_drones")}  maneuver={label.get("maneuver_type")}')
        print(f'    Detected: {det["detected"]}  prob={det["probability"]:.3f}')
        print(f'    Localised drones: {len(drones)}')
        for i, d in enumerate(drones):
            print(f'      [{i+1}] az={d["azimuth_deg"]:+6.1f}°  dist={d["distance_m"]:5.1f}m')

In [ ]:
# - 8-C: run_segment_analysis() in 3-mic multi-drone mode ---------─
# This demonstrates the full pipeline with multi_drone=True.

if _prof_dir_path and _prof_dir_path.exists():
    print('🔄 Running segment analysis with multi-drone mode...')
    md_results = run_segment_analysis(
        segment_dir  = str(_prof_dir_path),
        cfg          = config,
        file_ext     = ['.wav'],
        sort_by_name = True,
        n_mics       = 3,
        multi_drone  = True,
        show_plots   = True,
        save_plots   = True,
        save_csv     = True,
    )
    print(f'Multi-drone detections: {md_results["n_detected"]}/{md_results["n_segments"]}')
    print(f'Confirmed tracks: {len(md_results["confirmed_tracks"])}')
else:
    print('⚠️  Directory mode required - set PROF_MIC_SOURCE="dir" in 5-A.')

## 9 - Regression Dashboard (Aggregate Metrics)

In [ ]:
# - 9-A: Collect metrics from all sections -----------------─
import datetime

REGRESSION_TABLE = []

def _safe_mean(lst):
    vals = [v for v in lst if v is not None and not (isinstance(v, float) and math.isnan(v))]
    return round(sum(vals) / len(vals), 3) if vals else float('nan')

# Professional mic results
if PROF_MIC_RESULTS:
    REGRESSION_TABLE.append({
        'source':      'Dunakeszi BK-6-E (Professional)',
        'n_sessions':  PROF_MIC_RESULTS['n_sessions'],
        'n_labelled':  PROF_MIC_RESULTS['n_labelled'],
        'det_rate':    round(PROF_MIC_RESULTS['detection_rate'], 3),
        'mae_az_deg':  round(PROF_MIC_RESULTS['mae_az_deg'], 2),
        'mae_dist_m':  round(PROF_MIC_RESULTS['mae_dist_m'], 2),
        'mae_ht_m':    round(PROF_MIC_RESULTS['mae_ht_m'], 2),
    })

# MEMS results
if MEMS_RESULTS:
    n_mems_det = sum(1 for r in MEMS_RESULTS if r.get('detected'))
    REGRESSION_TABLE.append({
        'source':      'Dunakeszi MEMS (detection only)',
        'n_sessions':  len(MEMS_RESULTS),
        'n_labelled':  0,
        'det_rate':    round(n_mems_det / max(len(MEMS_RESULTS), 1), 3),
        'mae_az_deg':  float('nan'),
        'mae_dist_m':  float('nan'),
        'mae_ht_m':    float('nan'),
    })

# External audio results
if EXTERNAL_RESULTS:
    all_ext_segs = [seg for r in EXTERNAL_RESULTS for seg in r.get('segments', [])]
    n_ext_det    = sum(1 for s in all_ext_segs if s.get('detected'))
    REGRESSION_TABLE.append({
        'source':      f'External audio ({len(EXTERNAL_RESULTS)} file(s))',
        'n_sessions':  len(all_ext_segs),
        'n_labelled':  0,
        'det_rate':    round(n_ext_det / max(len(all_ext_segs), 1), 3),
        'mae_az_deg':  float('nan'),
        'mae_dist_m':  float('nan'),
        'mae_ht_m':    float('nan'),
    })

if not REGRESSION_TABLE:
    print('ℹ️  No results collected yet - run 5–8 first.')
else:
    df_reg = pd.DataFrame(REGRESSION_TABLE)
    print('\n' + '=' * 75)
    print('  REGRESSION DASHBOARD')
    print('=' * 75)
    display(df_reg.set_index('source'))

    # Save
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    csv_path = RESULTS_DIR / f'regression_{ts}.csv'
    df_reg.to_csv(csv_path, index=False)
    print(f'\n💾 Regression table saved: {csv_path}')

In [ ]:
# - 9-B: Visual comparison plot ----------------------─
import matplotlib.pyplot as plt

if REGRESSION_TABLE:
    df_plot = df_reg.dropna(subset=['mae_az_deg'])

    if df_plot.empty:
        print('ℹ️  No labelled sessions - skipping MAE comparison plot.')
    else:
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        fig.suptitle('Localisation MAE by Source', fontsize=13, fontweight='bold')

        for ax, (col, ylabel) in zip(axes, [
            ('mae_az_deg', 'MAE Azimuth (°)'),
            ('mae_dist_m', 'MAE Distance (m)'),
            ('mae_ht_m',   'MAE Height (m)'),
        ]):
            bars = ax.bar(df_plot['source'], df_plot[col], color='#1565c0', edgecolor='white', width=0.5)
            ax.set_ylabel(ylabel)
            ax.set_xticks(range(len(df_plot)))
            ax.set_xticklabels(df_plot['source'], rotation=20, ha='right', fontsize=8)
            ax.bar_label(bars, fmt='%.1f', fontsize=8, padding=2)
            ax.grid(axis='y', alpha=0.35)

        plt.tight_layout()
        plot_path = RESULTS_DIR / 'regression_comparison.png'
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'💾 Plot saved: {plot_path}')

    # Detection rate bar chart (all sources)
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    ax2.set_title('Detection Rate by Source', fontweight='bold')
    bars2 = ax2.bar(df_reg['source'], df_reg['det_rate'], color='#2e7d32', edgecolor='white', width=0.5)
    ax2.set_ylabel('Detection Rate')
    ax2.set_ylim(0, 1.1)
    ax2.axhline(0.8, color='orange', linestyle='--', alpha=0.7, label='80% threshold')
    ax2.bar_label(bars2, fmt='%.1%', fontsize=8, padding=2)
    ax2.set_xticks(range(len(df_reg)))
    ax2.set_xticklabels(df_reg['source'], rotation=20, ha='right', fontsize=8)
    ax2.grid(axis='y', alpha=0.35)
    ax2.legend()
    plt.tight_layout()
    fig2.savefig(RESULTS_DIR / 'detection_rate_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

print('\n✅ Regression dashboard complete.')

In [ ]:
# - 9-C: Full test summary printout --------------------─
print('=' * 70)
print('  FULL TEST SUMMARY')
print('=' * 70)
print(f'  Config : array={config.ARRAY_GEOMETRY}  MAX_LOC_DIST={config.MAX_LOCALIZATION_DIST}m')
print(f'  Device : {config.DEVICE}')
print(f'  BPF as feature: {config.BPF_ENERGY_RATIO_AS_FEATURE}')
print()

for row in REGRESSION_TABLE:
    print(f'  {row["source"]}')
    print(f'    Sessions : {row["n_sessions"]}  ({row["n_labelled"]} labelled)')
    print(f'    Det rate : {row["det_rate"]:.1%}')
    if not math.isnan(row['mae_az_deg']):
        print(f'    MAE az   : {row["mae_az_deg"]:.1f}°')
        print(f'    MAE dist : {row["mae_dist_m"]:.2f}m')
        print(f'    MAE ht   : {row["mae_ht_m"]:.2f}m')
    print()

print(f'  Results saved to: {RESULTS_DIR}')
print('=' * 70)